In [35]:
import pdfplumber
import pandas as pd
import re
import sqlite3
from langchain_community.utilities import SQLDatabase

In [36]:
tables = []

def clean_column_name(col):
    col = col.replace("\n", " ")
    col = col.strip()
    col = re.sub(r"[^\w\s]", "", col)
    col = col.replace(" ", "_")
    return col

with pdfplumber.open(r"data_files/Employee Performance.pdf") as pdf:

    for page_number, page in enumerate(pdf.pages):

        extracted_tables = page.extract_tables()

        for table_index, table in enumerate(extracted_tables):

            # convert to dataframe
            df = pd.DataFrame(
                table[1:],
                columns=table[0]
            )

            
            df.columns = [
                clean_column_name(col)
                for col in df.columns
            ]

            df = df.map(
                lambda x: x.replace("\n", " ") if isinstance(x, str) else x
            )

            tables.append({
                "page": page_number,
                "table_index": table_index,
                "dataframe": df
            })

In [37]:
tables[0]['dataframe']

,Department_ID,Department_Name,Manager_Name,Employees,Avg_Performance_Score,Monthly_Budget_
0,D101,Engineering,Ravi Narayanan,78,8.9,"24,00,000"
1,D102,Human Resources,Priya Sharma,16,8.1,"6,50,000"
2,D103,Marketing,Arjun Mehta,29,7.8,"11,00,000"
3,D104,Customer Support,Nisha Iyer,54,8.4,"13,50,000"
4,D105,Finance,Deepak Verma,18,8.7,"7,25,000"


In [38]:
conn = sqlite3.connect("pdf_data.db")
table_metadata = []

for table_info in tables:

    page = table_info["page"]

    table_index = table_info["table_index"]

    df = table_info["dataframe"]

    table_name = f"page_{page}_table_{table_index}"

    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )

    table_metadata.append({
        "table_name": table_name,
        "page": page,
        "columns": list(df.columns)
    })

    print(f"Inserted {table_name}")

Inserted page_0_table_0
Inserted page_1_table_0
Inserted page_1_table_1
Inserted page_2_table_0
Inserted page_4_table_0


In [39]:
table_metadata[0]

{'table_name': 'page_0_table_0',
 'page': 0,
 'columns': ['Department_ID',
  'Department_Name',
  'Manager_Name',
  'Employees',
  'Avg_Performance_Score',
  'Monthly_Budget_']}

In [46]:
tables[2]['dataframe']

,Asset_ID,Asset_Type,Assigned_Department,Status,Last_Maintenance_Date
0,AS001,Dell PowerEdge R760 Server,Engineering,Active,2026-01-14
1,AS002,Cisco Catalyst Switch,Engineering,Active,2026-02-01
2,AS003,HP LaserJet Pro Printer,Human Resources,Active,2025-12-18
3,AS004,Lenovo ThinkStation Workstation,Marketing,Under Repair,2026-03-05
4,AS005,Synology NAS Storage,Finance,Active,2026-01-30


In [49]:
table_metadata[1]

{'table_name': 'page_1_table_0',
 'page': 1,
 'columns': ['Employee_ID',
  'Employee_Name',
  'Department',
  'Role',
  'Experience_Years',
  'Performance_Score',
  'Salary_']}